# Manipulating models

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:

1. Make temporary, automatically reverted changes to a model with `with model:`.
2. Choose between `model.optimize()` and `model.slim_optimize()` for speed.
3. Define growth media by editing `model.medium` and explain the sign convention of exchange reactions.
4. Add new metabolites and reactions to a model, and use exchange, demand and sink reactions correctly.

**Estimated time:** 50 min (20 min guided, 30 min exercises).

**Prerequisites:** notebooks 01 and 02.

### Background

**Copies versus contexts.** Many analyses change a model over and over, e.g. knocking out each gene in turn. Copying a genome-scale model for every change is slow. cobrapy ([Ebrahim et al., 2013](https://doi.org/10.1186/1752-0509-7-74)) instead records every change made inside a `with model:` block and undoes them when the block ends, so the model is always back in its original state.

**Boundary reactions.** A metabolic model is a system with borders. Metabolites can only enter or leave through *boundary reactions*, which have a single metabolite and no product:

| Type | Typical use | Written as | Reversible? |
|---|---|---|---|
| Exchange (`EX_`) | uptake or secretion of an extracellular metabolite | `glc__D_e <=>` | yes |
| Demand (`DM_`) | drain of an intracellular metabolite, e.g. a product we want to measure | `gold_c -->` | no |
| Sink (`SK_`) | supply *and* drain of an intracellular metabolite that the model cannot make | `met_c <=>` | yes |

Because exchange reactions are written as `metabolite <=>` (nothing), a **negative flux means uptake** and a positive flux means secretion. `model.medium` hides this sign: it lists the *maximal uptake rates* as positive numbers (the negated lower bounds of the exchange reactions).

The examples use the *E. coli* core model, a small model of central carbon metabolism made for teaching ([Orth et al., 2010](https://doi.org/10.1128/ecosalplus.10.2.1)).

In [ ]:
from cobra.io import read_sbml_model
model = read_sbml_model('data/e_coli_core.xml.gz')

## Making temporary changes to the model

The obvious way to change a model while keeping the original is to make a copy. Copying is slow, though. `%%time` measures how long a cell takes to run.

In [ ]:
model.optimize()

In [ ]:
%%time
copy_of_model = model.copy()

Even milliseconds add up quickly when you run many simulations, for example when knocking out every gene individually to check whether it is essential. The next cell copies the model once per gene:

In [ ]:
%%time
for gene in model.genes:
    mutant = model.copy()
    mutant.genes.get_by_id(gene.id).knock_out()

For that reason cobrapy provides a faster mechanism. Almost all methods that change the model (knocking out genes or reactions, adding or removing metabolites and reactions, changing bounds or the objective) are automatically reverted when you leave a `with model:` block. Compare the run time with the cell above:

In [ ]:
%%time
with model:
    for gene in model.genes:
        gene.knock_out()

The `with model:` statement starts the context, and every change made in the indented block is recorded. When the block finishes, all changes are rolled back, leaving the model exactly as it was before.

Changing flux bounds is also reverted. For example, let us set the lower and upper bound of phosphoglycerate kinase (PGK) to 0, which effectively knocks out the reaction:

In [ ]:
with model:
    model.reactions.PGK.bounds = 0, 0
    print("PGK's bounds inside the with statement")
    print(model.reactions.PGK.lower_bound, model.reactions.PGK.bounds)
    print('Mutant growth rate: ', model.optimize().objective_value)
print("PGK's bounds outside the with statement")
print(model.reactions.PGK.bounds)

## Slim versus full optimize

Solvers are so fast that for small and mid-size models, collecting the results and converting them into Python objects can take longer than the optimization itself. `model.optimize()` gathers the values of all reactions and metabolites into a `Solution` object. If you only need the objective value or a few fluxes, `model.slim_optimize()` is faster: it only optimizes and returns the objective value, and you read other values yourself (e.g. `model.reactions.ATPM.flux`). For example, let us get the flux of the ATP maintenance reaction (`ATPM`) both ways:

In [ ]:
%%time
solution = model.optimize()
solution.fluxes['ATPM']

In [ ]:
%%time
model.slim_optimize()
model.reactions.ATPM.flux

The difference may seem small, but over thousands of simulations it adds up.

## Changing the medium

The medium is available as `model.medium`: a dictionary from exchange reaction to its maximal uptake rate in mmol gDW⁻¹ h⁻¹. Here glucose uptake is limited to 10, while water, protons, CO₂, ammonium, phosphate and oxygen are essentially unlimited (1000).

In [ ]:
model.medium

To change the carbon source, edit the dictionary and assign it back to `model.medium` (assigning is required; editing the dictionary alone does nothing). The following block removes glucose and adds succinate, inside a `with` block so the change is undone afterwards.

In [ ]:
medium = model.medium
with model:
    medium['EX_glc__D_e'] = 0
    medium['EX_succ_e'] = 10
    model.medium = medium
    solution = model.optimize()
    print(solution.fluxes['BIOMASS_Ecoli_core_w_GAM'])

Growth on succinate is much slower than on glucose: succinate has fewer carbon atoms (4 versus 6), and it enters metabolism in the TCA cycle, so gluconeogenesis is needed to make sugar-derived biomass precursors.

## Exercise (10 min)

* Change the carbon source in the medium to a different carbon source. What is the difference in the growth rate observed?
* How about growing E. coli under anaerobic conditions?

In [ ]:
# Your code here


## Adding reactions and pathways

In [ ]:
from cobra import Reaction, Metabolite

Let's create a new reaction. A reaction needs an identifier; name, bounds and metabolites can be added afterwards.

In [ ]:
new_reaction = Reaction('alchemy')

This reaction is going to convert water into gold (unfortunately lead is not part of _E. coli_ metabolism; creating wine would be blasphemy). Gold is not part of _E. coli's_ native metabolism, so we first create a new metabolite in the cytosol (`c`).

In [ ]:
gold = Metabolite(id='gold_c', compartment='c')

Now we specify the reaction's stoichiometry: negative coefficients are substrates, positive coefficients are products.

In [ ]:
new_reaction.add_metabolites({model.metabolites.h2o_c: -1, gold: 1})

Printing the reaction reveals that the reaction indeed converts water into gold.

In [ ]:
print(new_reaction.build_reaction_string())

Now, let's add the new reaction to the model.

In [ ]:
model.add_reactions([new_reaction])

Quickly check that the reaction was indeed added to the model.

In [ ]:
model.reactions.alchemy

Let's produce some gold then!

In [ ]:
model.objective = model.reactions.alchemy
model.optimize().objective_value

:-(

What happened? Gold is produced but has nowhere to go. At steady state ($Sv = 0$) every metabolite must be consumed as fast as it is produced, so a dead-end metabolite forces the flux of its producing reaction to zero. We need a boundary reaction that lets gold leave the system: a *demand* reaction.

In [ ]:
model.add_boundary(model.metabolites.gold_c, type='demand')

In [ ]:
model.objective = model.reactions.alchemy
model.optimize().objective_value

Much better! (Our alchemy reaction is of course not mass balanced; `new_reaction.check_mass_balance()` would tell you so. Real pathways should always be checked.)

## Exercise (20 min)

* Add a pathway to the model (ideally one that you're personally interested in; you can also use a different model if you like). 

In [ ]:
# Your code here


## References

- Ebrahim, Lerman, Palsson et al. (2013). COBRApy: COnstraints-Based Reconstruction and Analysis for Python. *BMC Systems Biology* 7:74. [doi.org/10.1186/1752-0509-7-74](https://doi.org/10.1186/1752-0509-7-74)
- Orth, Fleming, Palsson (2010). Reconstruction and Use of Microbial Metabolic Networks: the Core *Escherichia coli* Metabolic Model as an Educational Guide. *EcoSal Plus* 4. [doi.org/10.1128/ecosalplus.10.2.1](https://doi.org/10.1128/ecosalplus.10.2.1)